# PSU Eco Racing - Segformer Fine-Tune
Before running: **Runtime > Change runtime type > T4 GPU**

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
assert torch.cuda.is_available(), 'Switch to GPU runtime first!'

In [ ]:
!pip install -q roboflow transformers==4.40.0 pycocotools torchvision

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="I80CqqBjjzKuReCGqmo9")
project = rf.workspace("taynamghzs-workspace").project("semantic-segmentation-zntjb")
version = project.version(2)
dataset = version.download("coco-segmentation")
DATA_ROOT = dataset.location
print('Dataset at:', DATA_ROOT)

In [ ]:
import os
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util

def convert_split(split):
    img_dir  = os.path.join(DATA_ROOT, split)
    ann_file = os.path.join(img_dir, '_annotations.coco.json')
    mask_dir = os.path.join(DATA_ROOT, split + '_masks')
    os.makedirs(mask_dir, exist_ok=True)
    if not os.path.exists(ann_file):
        print('No annotations for', split)
        return mask_dir
    coco = COCO(ann_file)
    for img_id, img_info in coco.imgs.items():
        h, w  = img_info['height'], img_info['width']
        mask  = np.zeros((h, w), dtype=np.uint8)
        for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
            rle = coco_mask_util.frPyObjects(ann['segmentation'], h, w)
            m   = coco_mask_util.decode(rle)
            if m.ndim == 3:
                m = m.max(axis=2)
            mask[m > 0] = 1
        fname = os.path.splitext(img_info['file_name'])[0] + '.png'
        Image.fromarray(mask).save(os.path.join(mask_dir, fname))
    print(split, ':', len(coco.imgs), 'masks saved')
    return mask_dir

train_mask_dir = convert_split('train')
valid_mask_dir = convert_split('valid')

In [ ]:
import random
import torch
from pathlib import Path
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from transformers import SegformerImageProcessor

ROAD_CLASS = 0
IGNORE_IDX = 255
IMG_SIZE   = (512, 512)
MODEL_ID   = 'nvidia/segformer-b2-finetuned-cityscapes-1024-1024'

processor = SegformerImageProcessor.from_pretrained(MODEL_ID)

class TrackDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_dir  = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        mask_stems    = {p.stem for p in self.mask_dir.glob('*.png')}
        self.imgs     = sorted([p for p in self.img_dir.iterdir()
                                if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
                                and p.stem in mask_stems])
        self.augment  = augment
        print(img_dir, ':', len(self.imgs), 'images')

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img  = Image.open(self.imgs[idx]).convert('RGB').resize(IMG_SIZE)
        mask = Image.open(self.mask_dir / (self.imgs[idx].stem + '.png'))
        mask = mask.resize(IMG_SIZE, resample=Image.NEAREST)
        if self.augment:
            if random.random() > 0.5:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)
            img = TF.adjust_brightness(img, 0.6 + random.random() * 0.8)
            img = TF.adjust_contrast(img,   0.6 + random.random() * 0.8)
            img = TF.adjust_saturation(img, 0.6 + random.random() * 0.8)
        m_np  = np.array(mask, dtype=np.int64)
        label = np.full_like(m_np, IGNORE_IDX)
        label[m_np == 1] = ROAD_CLASS
        enc = processor(images=img, return_tensors='pt')
        return enc['pixel_values'].squeeze(0), torch.tensor(label, dtype=torch.long)

train_ds = TrackDataset(os.path.join(DATA_ROOT, 'train'), train_mask_dir, augment=True)
valid_ds = TrackDataset(os.path.join(DATA_ROOT, 'valid'), valid_mask_dir, augment=False)

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import SegformerForSemanticSegmentation

EPOCHS     = 30
BATCH_SIZE = 4
LR         = 6e-5

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID).cuda().train()

optimizer = AdamW([
    {'params': model.segformer.parameters(),   'lr': LR * 0.1},
    {'params': model.decode_head.parameters(), 'lr': LR},
], weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_loss = float('inf')
os.makedirs('/content/best_model', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss = 0.0
    for pv, labels in train_dl:
        pv, labels = pv.cuda(), labels.cuda()
        H, W    = labels.shape[-2], labels.shape[-1]
        logits  = model(pixel_values=pv).logits
        up      = F.interpolate(logits, (H, W), mode='bilinear', align_corners=False)
        loss    = F.cross_entropy(up, labels, ignore_index=IGNORE_IDX)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_dl)

    model.eval()
    v_loss = iou_sum = 0.0
    with torch.no_grad():
        for pv, labels in valid_dl:
            pv, labels = pv.cuda(), labels.cuda()
            H, W   = labels.shape[-2], labels.shape[-1]
            logits = model(pixel_values=pv).logits
            up     = F.interpolate(logits, (H, W), mode='bilinear', align_corners=False)
            v_loss += F.cross_entropy(up, labels, ignore_index=IGNORE_IDX).item()
            pred   = up.argmax(1)
            valid  = labels != IGNORE_IDX
            tp = ((pred == 0) & (labels == 0) & valid).sum().item()
            fp = ((pred == 0) & (labels != 0) & valid).sum().item()
            fn = ((pred != 0) & (labels == 0) & valid).sum().item()
            iou_sum += tp / (tp + fp + fn + 1e-6)
    v_loss   /= len(valid_dl)
    road_iou  = iou_sum / len(valid_dl)
    scheduler.step()
    print(f'Epoch {epoch:3d}/{EPOCHS}  train={t_loss:.4f}  val={v_loss:.4f}  road_IoU={road_iou:.3f}')
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        model.save_pretrained('/content/best_model')
        processor.save_pretrained('/content/best_model')
        print('  ^ best saved')

print('Done. Best val loss:', round(best_val_loss, 4))

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/psu_segformer_best', 'zip', '/content/best_model')
files.download('/content/psu_segformer_best.zip')
print('Done - unzip and copy to Jetson, then set SEG_MODEL_ID in config.py')